In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [2]:
import os # 모델 저장할 폴더가 있는지 여부와 폴더 생성
import numpy as np
import pandas as pd # get_dummies(원핫인코딩), crosstab
import seaborn as sns # iris데이터(데이터프레임) 가져오기
from sklearn import datasets # iris데이터(X,y) 가져오기
from sklearn.preprocessing import LabelEncoder # 라벨인코더
from tensorflow.keras.utils import to_categorical # 원핫인코딩
from sklearn.preprocessing import StandardScaler # , MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, save_model, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, LeakyReLU
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
import matplotlib.pyplot as plt

# 1. 기본적인 DNN

In [26]:
# 1. 데이터 생성 및 전처리
iris = datasets.load_iris()
iris_X = iris.data
iris_y = iris.target
train_X, test_X, train_y, test_y = train_test_split(iris_X, iris_y,
                                                   train_size=0.8,
                                                   stratify=iris_y)

## 모델 구성

In [27]:
model = Sequential([
    Input(4),
    Dense(units=64, activation='relu'),
    Dense(units=128, activation='relu'),
    Dense(units=50, activation='relu'),
    Dense(units=30, activation='relu'),
    Dense(units=3, activation='softmax'),
])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_15 (Dense)            (None, 64)                320       
                                                                 
 dense_16 (Dense)            (None, 128)               8320      
                                                                 
 dense_17 (Dense)            (None, 50)                6450      
                                                                 
 dense_18 (Dense)            (None, 30)                1530      
                                                                 
 dense_19 (Dense)            (None, 3)                 93        
                                                                 
Total params: 16,713
Trainable params: 16,713
Non-trainable params: 0
_________________________________________________________________


## 학습과정 설정

In [28]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])

## 모델 학습

In [30]:
earlyStopping = EarlyStopping(monitor='val_loss', patience=10)
hist = model.fit(train_X, train_y, epochs=200,
                validation_split=0.1,
                verbose=1,
                callbacks=[earlyStopping])

Epoch 1/200
4/4 [==============================] - 0s 16ms/step - loss: 0.0610 - accuracy: 0.9722 - val_loss: 8.0220e-04 - val_accuracy: 1.0000
Epoch 2/200
4/4 [==============================] - 0s 10ms/step - loss: 0.0659 - accuracy: 0.9815 - val_loss: 7.0571e-04 - val_accuracy: 1.0000
Epoch 3/200
4/4 [==============================] - 0s 10ms/step - loss: 0.0580 - accuracy: 0.9815 - val_loss: 0.0032 - val_accuracy: 1.0000
Epoch 4/200
4/4 [==============================] - 0s 9ms/step - loss: 0.0986 - accuracy: 0.9630 - val_loss: 0.0016 - val_accuracy: 1.0000
Epoch 5/200
4/4 [==============================] - 0s 8ms/step - loss: 0.0705 - accuracy: 0.9815 - val_loss: 9.5490e-04 - val_accuracy: 1.0000
Epoch 6/200
4/4 [==============================] - 0s 4ms/step - loss: 0.0772 - accuracy: 0.9537 - val_loss: 0.0025 - val_accuracy: 1.0000
Epoch 7/200
4/4 [==============================] - 0s 6ms/step - loss: 0.0732 - accuracy: 0.9722 - val_loss: 0.0028 - val_accuracy: 1.0000
Epoch 8/200


# 2. sklearn이용
- 원핫인코딩을 하지 않고 라벨인코딩까지만 해야 작동

In [32]:
from sklearn.neural_network import MLPClassifier

In [31]:
# 1. 데이터
train_X.shape, train_y.shape, test_X.shape, test_y.shape

((120, 4), (120,), (30, 4), (30,))

In [33]:
model = MLPClassifier(
            hidden_layer_sizes=(64,128,50), # hidden layer의 units 수
            activation='relu',
            solver= 'adam', # sgd등의 optimizer
            batch_size=40,
            max_iter=1000, # 학습최대횟수
            early_stopping=True, # 조기 종료 활성화
            n_iter_no_change=10, # early_stopping의 patience와 유사
            validation_fraction=0.1, # 검증셋 비율
            warm_start=False # True일 경우 이전 학습에 이어서 학습
)

In [34]:
model.fit(train_X,train_y)

MLPClassifier(batch_size=40, early_stopping=True,
              hidden_layer_sizes=(64, 128, 50), max_iter=1000)

In [35]:
model.score(test_X,test_y)

0.8666666666666667

In [36]:
iris_X[0]

array([5.1, 3.5, 1.4, 0.2])

In [37]:
# 모델 사용
input_data=[[5.1, 3.5, 1.4, 0.2]]
model.predict(input_data)

array([0])

In [40]:
# 교차표
# test_y : 실제값
hat_y = model.predict(test_X) 
pd.crosstab(test_y, hat_y, rownames=['real'], colnames=['predict'])

predict,0,1,2
real,,,
0,10,0,0
1,0,7,3
2,0,1,9


# 3. 클래스를 생성하여 모델 생성함수 사용

In [44]:
class DNNClassifier:
    @staticmethod
    def build(input_dim=4, activation='relu'):
        # 모델구성
        model = Sequential([
            Input(input_dim),
            Dense(units=50, activation=activation),
            Dense(units=30, activation=activation),
            Dense(units=3, activation='softmax')
        ])
        # 학습설정
        model.compile(loss='sparse_categorical_crossentropy',
                     optimizer='adam',
                     metrics=['accuracy'])
        return model

In [47]:
# 1. 데이터
print(train_X.shape, train_y.shape, test_X.shape, test_y.shape)
# 2. 모델
model = DNNClassifier.build(input_dim = 4, activation='elu')
hist=model.fit(train_X, train_y, epochs=50, validation_split=0.1)

(120, 4) (120,) (30, 4) (30,)
Epoch 1/50
4/4 [==============================] - 0s 52ms/step - loss: 1.3724 - accuracy: 0.3519 - val_loss: 1.6736 - val_accuracy: 0.0833
Epoch 2/50
4/4 [==============================] - 0s 6ms/step - loss: 1.1954 - accuracy: 0.2778 - val_loss: 1.3999 - val_accuracy: 0.0833
Epoch 3/50
4/4 [==============================] - 0s 6ms/step - loss: 1.0876 - accuracy: 0.3519 - val_loss: 1.1857 - val_accuracy: 0.0833
Epoch 4/50
4/4 [==============================] - 0s 8ms/step - loss: 1.0182 - accuracy: 0.3981 - val_loss: 1.0422 - val_accuracy: 0.5833
Epoch 5/50
4/4 [==============================] - 0s 5ms/step - loss: 0.9634 - accuracy: 0.6296 - val_loss: 0.9764 - val_accuracy: 0.6667
Epoch 6/50
4/4 [==============================] - 0s 6ms/step - loss: 0.9069 - accuracy: 0.6667 - val_loss: 0.9080 - val_accuracy: 0.6667
Epoch 7/50
4/4 [==============================] - 0s 11ms/step - loss: 0.8487 - accuracy: 0.6944 - val_loss: 0.8457 - val_accuracy: 0.8333
Ep

In [48]:
# 모델 평가
loss, accuracy = model.evaluate(test_X,test_y)
print(f'정확도 : {accuracy*100}%')

1/1 [==============================] - 0s 27ms/step - loss: 0.1416 - accuracy: 0.9667
정확도 : 96.66666388511658%


# 함수형 API
- 병렬처리, Residual block

In [ ]:
# 기존 model 스타일1
# 기존 model 스타일2
model = Sequential([
            Input(shape=(4,)),
            Dense(units=50, activation='relu'),
            Dense(units=30, activation='relu'),
            Dense(units=3, activation='softmax')
])

In [52]:
# 기존 model 스타일3(함수형 API)
from tensorflow.keras import Model
input_ = Input(shape=(4,))
layer1 = Dense(units=50, activation='relu')(input_)
layer2 = Dense(units=30, activation='relu')(layer1)
output = Dense(units=3, activation='softmax')(layer2)
model = Model(inputs=input_, outputs=output)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_9 (InputLayer)        [(None, 4)]               0         
                                                                 
 dense_32 (Dense)            (None, 50)                250       
                                                                 
 dense_33 (Dense)            (None, 30)                1530      
                                                                 
 dense_34 (Dense)            (None, 3)                 93        
                                                                 
Total params: 1,873
Trainable params: 1,873
Non-trainable params: 0
_________________________________________________________________


In [53]:
# 병렬처리 방식
from tensorflow.keras.layers import concatenate
input_ = Input(shape=(4,))
dense1 = Dense(units=50, activation='relu')(input_)
dense2 = Dense(units=80, activation='relu')(input_)
dense3 = Dense(units=30, activation='relu')(input_)
x = concatenate([dense1, dense2, dense3])
output = Dense(units=3, activation='softmax')(x)
model= Model(inputs=input_, outputs= output)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 4)]          0           []                               
                                                                                                  
 dense_35 (Dense)               (None, 50)           250         ['input_10[0][0]']               
                                                                                                  
 dense_36 (Dense)               (None, 80)           400         ['input_10[0][0]']               
                                                                                                  
 dense_37 (Dense)               (None, 30)           150         ['input_10[0][0]']               
                                                                                            

In [55]:
# 리지듀얼 블럭(Residual block): 딥러닝에서 기울기 소실 문제로 학습이 잘 되지 않는 부분을 해결하기 위한 제안
from tensorflow.keras.layers import add
input_ = Input(shape=(4,))
dense1 = Dense(units=50, activation='relu')(input_)
dense2 = Dense(units=50, activation='relu')(dense1)
dense3 = add([dense1,dense2])
output = Dense(units=3, activation='softmax')(dense3)
model=Model(inputs=input_, outputs=output)
model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_11 (InputLayer)          [(None, 4)]          0           []                               
                                                                                                  
 dense_39 (Dense)               (None, 50)           250         ['input_11[0][0]']               
                                                                                                  
 dense_40 (Dense)               (None, 50)           2550        ['dense_39[0][0]']               
                                                                                                  
 add (Add)                      (None, 50)           0           ['dense_39[0][0]',               
                                                                  'dense_40[0][0]']         

In [56]:
# 학습과정 설정 & 학습
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='adam', metrics=['accuracy'])
model.fit(train_X,train_y, epochs=100, verbose=1)

Epoch 1/100
4/4 [==============================] - 1s 3ms/step - loss: 2.5327 - accuracy: 0.3333
Epoch 2/100
4/4 [==============================] - 0s 727us/step - loss: 2.0108 - accuracy: 0.5833
Epoch 3/100
4/4 [==============================] - 0s 5ms/step - loss: 1.6211 - accuracy: 0.6667
Epoch 4/100
4/4 [==============================] - 0s 0s/step - loss: 1.2363 - accuracy: 0.6667
Epoch 5/100
4/4 [==============================] - 0s 1ms/step - loss: 0.9437 - accuracy: 0.6667
Epoch 6/100
4/4 [==============================] - 0s 1ms/step - loss: 0.7742 - accuracy: 0.6417
Epoch 7/100
4/4 [==============================] - 0s 732us/step - loss: 0.7415 - accuracy: 0.6667
Epoch 8/100
4/4 [==============================] - 0s 5ms/step - loss: 0.7299 - accuracy: 0.6667
Epoch 9/100
4/4 [==============================] - 0s 0s/step - loss: 0.6962 - accuracy: 0.6667
Epoch 10/100
4/4 [==============================] - 0s 2ms/step - loss: 0.6453 - accuracy: 0.6667
Epoch 11/100
4/4 [=========

4/4 [==============================] - 0s 4ms/step - loss: 0.1494 - accuracy: 0.9667
Epoch 85/100
4/4 [==============================] - 0s 0s/step - loss: 0.1486 - accuracy: 0.9583
Epoch 86/100
4/4 [==============================] - 0s 5ms/step - loss: 0.1452 - accuracy: 0.9667
Epoch 87/100
4/4 [==============================] - 0s 0s/step - loss: 0.1477 - accuracy: 0.9667
Epoch 88/100
4/4 [==============================] - 0s 1ms/step - loss: 0.1445 - accuracy: 0.9667
Epoch 89/100
4/4 [==============================] - 0s 0s/step - loss: 0.1479 - accuracy: 0.9583
Epoch 90/100
4/4 [==============================] - 0s 1ms/step - loss: 0.1387 - accuracy: 0.9667
Epoch 91/100
4/4 [==============================] - 0s 3ms/step - loss: 0.1413 - accuracy: 0.9667
Epoch 92/100
4/4 [==============================] - 0s 333us/step - loss: 0.1399 - accuracy: 0.9583
Epoch 93/100
4/4 [==============================] - 0s 4ms/step - loss: 0.1352 - accuracy: 0.9750
Epoch 94/100
4/4 [================

In [57]:
# 학습 평가
loss, accurcy = model.evaluate(test_X, test_y, verbose=0)
print(f'accuracy:{accuracy}')

accuracy:0.9666666388511658
